# Total Energy + Post Processing

This notebook demonstrates how to run a Total Energy calculation with optional post-processing (PP) steps using the Mat3ra API and Quantum ESPRESSO.

## Process Overview
1. Set up environment and parameters
2. Authenticate and initialize API client
3. Select account and project
4. Load and save material
5. Configure workflow with selected PP property
6. Set up compute resources
7. Create and submit job
8. Retrieve and visualize results

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
import os
import sys

if sys.platform == "emscripten":
    import micropip

    await micropip.install("mat3ra-api-examples", deps=False)
    await micropip.install("mat3ra-utils")
    from mat3ra.utils.jupyterlite.packages import install_packages

    await install_packages("api_examples")

### 1.2. Set parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
MATERIAL_NAME = "Silicon"

# 4. Workflow parameters
APPLICATION_NAME = "espresso"
ADD_RELAXATION = False

# PP property to calculate. Options:
#   "wavefunction_amplitude"          - wavefunction amplitude (pp.x, plot_num=7)
#   "electronic_density_mesh"         - electronic charge density mesh (pp.x, plot_num=0)
#   "average_electrostatic_potential" - planar-averaged electrostatic potential (pp.x + average.x)
#   "ldos"                            - local density of states (pp.x, plot_num=3)
#   None                              - total energy only, no PP
PP_PROPERTY = "wavefunction_amplitude"

# --- wavefunction_amplitude parameters ---
# Band selection: "VBM" (valence band maximum), "CBM" (conduction band minimum),
#                 or an integer (1-based custom band index)
WFN_BAND = "VBM"
# Plot plane/direction: "XY", "XZ", "YZ", or "Z" (1D along Z)
WFN_PLANE = "XY"

# --- average_electrostatic_potential parameters ---
AVG_SAMPLING_SCALE = 1.0   # macroscopic average period factor
AVG_NUM_POINTS = 3000      # number of points along the averaging direction
AVG_SMOOTHING_WINDOW = 3.0 # smoothing window length (in Bohr)

# --- ldos parameters ---
LDOS_EMIN = -5.0   # lower energy bound relative to Fermi level (eV)
LDOS_EMAX = 5.0    # upper energy bound relative to Fermi level (eV)
LDOS_DELTA_E = 0.1 # energy step for LDOS (eV)

SAVE_WF_TO_COLLECTION = True

# 5. Compute parameters
CLUSTER_NAME = None
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
JOB_NAME = f"TE+PP ({PP_PROPERTY}) {timestamp}" if PP_PROPERTY else f"Total Energy {timestamp}"
MY_WORKFLOW_NAME = f"Total Energy + {PP_PROPERTY.replace('_', ' ').title()}" if PP_PROPERTY else "Total Energy"
POLL_INTERVAL = 30  # seconds

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Create material
### 3.1. Load material from local file (or Standata)

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from utils.visualize import visualize_materials as visualize
from utils.jupyterlite import load_material_from_folder

material = load_material_from_folder(FOLDER, MATERIAL_NAME) or Material.create(
    Materials.get_by_name_first_match(MATERIAL_NAME))

visualize(material)

### 3.2. Save material to the platform

In [ ]:
from utils.api import get_or_create_material
from utils.generic import dict_to_namespace

saved_material_response = get_or_create_material(client.materials, material, ACCOUNT_ID)
saved_material = dict_to_namespace(saved_material_response)

## 4. Configure workflow
### 4.1. Select application

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.ade.application import Application

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")

### 4.2. Load base workflow and find target subworkflow

Each PP property maps to a dedicated standata subworkflow that already includes the SCF step.
For `ldos` (not in standata), PP units are added to the Total Energy subworkflow.

In [ ]:
from mat3ra.standata.subworkflows import SubworkflowStandata
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.wode.subworkflows import Subworkflow
from utils.visualize import visualize_workflow

PP_SUBWORKFLOW_NAME_MAP = {
    "wavefunction_amplitude": "wavefunction_amplitude.json",
    "electronic_density_mesh": "electronic_density_mesh.json",
    "average_electrostatic_potential": "average_electrostatic_potential.json",
    "ldos": "ldos.json",
}

if PP_PROPERTY and PP_PROPERTY in PP_SUBWORKFLOW_NAME_MAP:
    swf_config = SubworkflowStandata.filter_by_application(app.name).get_by_name_first_match(
        PP_SUBWORKFLOW_NAME_MAP[PP_PROPERTY]
    )
    base_subworkflow = Subworkflow(**swf_config)
    base_subworkflow.application = app
    print(f"✅ Using standata subworkflow: {base_subworkflow.name}")
else:
    te_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match("total_energy.json")
    base_subworkflow = Subworkflow(**te_config["subworkflows"][0])
    print("✅ Using Total Energy subworkflow (no PP)")

workflow = Workflow(name=MY_WORKFLOW_NAME)
workflow.add_subworkflow(base_subworkflow)
visualize_workflow(workflow)

### 4.3. Adjust PP step parameters
#### 4.3.1. Wavefunction amplitude: band selection and plot plane

In [ ]:
if PP_PROPERTY == "wavefunction_amplitude":
    swf = workflow.subworkflows[0]

    # Adjust band selection in the "Select Band" assignment unit
    select_band_unit = swf.get_unit_by_name(name="Select Band")
    if select_band_unit is not None:
        if WFN_BAND == "VBM":
            select_band_unit.value = "KBAND_VALUE_BELOW_EF"
        elif WFN_BAND == "CBM":
            select_band_unit.value = "KBAND_VALUE_ABOVE_EF"
        else:
            select_band_unit.value = str(int(WFN_BAND))
        swf.set_unit(select_band_unit)
        print(f"✅ Band selection set to: {WFN_BAND} (value: {select_band_unit.value})")

    # Adjust plot plane in pp_wfn.in input template (inputs are dicts)
    WFN_PLANE_VECTORS = {
        "XY": ("iflag = 2", "e1(1) = 1.0, e1(2) = 0.0, e1(3) = 0.0", "e2(1) = 0.0, e2(2) = 1.0, e2(3) = 0.0"),
        "XZ": ("iflag = 2", "e1(1) = 1.0, e1(2) = 0.0, e1(3) = 0.0", "e2(1) = 0.0, e2(2) = 0.0, e2(3) = 1.0"),
        "YZ": ("iflag = 2", "e1(1) = 0.0, e1(2) = 1.0, e1(3) = 0.0", "e2(1) = 0.0, e2(2) = 0.0, e2(3) = 1.0"),
        "Z":  ("iflag = 1", "e1(1) = 0.0, e1(2) = 0.0, e1(3) = 1.0", None),
    }
    pp_wfn_unit = swf.get_unit_by_name(name="pp_wfn")
    if pp_wfn_unit is not None and WFN_PLANE in WFN_PLANE_VECTORS:
        iflag_str, e1_str, e2_str = WFN_PLANE_VECTORS[WFN_PLANE]
        for inp in pp_wfn_unit.input:
            content = inp["rendered"]
            content = content.replace("iflag = 1", iflag_str).replace("iflag = 2", iflag_str)
            content = content.replace("e1(1) = 0.0, e1(2) = 0.0, e1(3) = 1.0", e1_str)
            if e2_str and "e2(" not in content:
                content = content.replace("nx = 200", f"{e2_str}\n    nx = 200\n    ny = 200")
            inp["rendered"] = content
        swf.set_unit(pp_wfn_unit)
        print(f"✅ Plot plane set to: {WFN_PLANE}")

#### 4.3.2. Average electrostatic potential: sampling scale, number of points, smoothing window

In [ ]:
if PP_PROPERTY == "average_electrostatic_potential":
    swf = workflow.subworkflows[0]
    avg_unit = swf.get_unit_by_name(name="average ESP")
    if avg_unit is not None:
        # average.in format (line-based): filein, filplot, weight, npts, idir, alat
        new_average_in = f"1\npp.dat\n{AVG_SAMPLING_SCALE}\n{AVG_NUM_POINTS}\n3\n{AVG_SMOOTHING_WINDOW}\n"
        for inp in avg_unit.input:
            inp["rendered"] = new_average_in
        swf.set_unit(avg_unit)
        print(
            f"✅ average.in updated: scale={AVG_SAMPLING_SCALE}, "
            f"npoints={AVG_NUM_POINTS}, window={AVG_SMOOTHING_WINDOW}"
        )

#### 4.3.3. LDOS: adjust energy window in standata subworkflow

In [ ]:
if PP_PROPERTY == "ldos":
    swf = workflow.subworkflows[0]
    pp_ldos_unit = swf.get_unit_by_name(name="pp_ldos")
    if pp_ldos_unit is not None:
        for inp in pp_ldos_unit.input:
            content = inp["rendered"]
            content = content.replace("emin = -5.0", f"emin = {LDOS_EMIN}")
            content = content.replace("emax = 5.0", f"emax = {LDOS_EMAX}")
            inp["rendered"] = content
        swf.set_unit(pp_ldos_unit)
        print(f"✅ LDOS energy window set: emin={LDOS_EMIN} eV, emax={LDOS_EMAX} eV (relative to Fermi)")

### 4.4. Add relaxation (optional)

In [ ]:
if ADD_RELAXATION:
    workflow.add_relaxation()
    visualize_workflow(workflow)

### 4.5. Modify model and k-grid (optional)

In [ ]:
# # Example: Change model parameters
#
# from mat3ra.mode import Model
# from mat3ra.standata.model_tree import ModelTreeStandata
#
# model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype="gga", functional="pbe")
# model_config["method"] = {"type": "pseudopotential", "subtype": "us"}
# model = Model.create(model_config)
# for swf in workflow.subworkflows:
#     swf.model = model

# # Example: Change k-grid
#
# from mat3ra.wode.context.providers import PointsGridDataProvider
# SCF_KGRID = [4, 4, 4]
# new_context_scf = PointsGridDataProvider(dimensions=SCF_KGRID, isEdited=True).yield_data()
#
# pp_swf = workflow.subworkflows[1 if ADD_RELAXATION else 0]
# unit_scf = pp_swf.get_unit_by_name(name="pw_scf")
# unit_scf.add_context(new_context_scf)
# pp_swf.set_unit(unit_scf)
# visualize_workflow(workflow)

### 4.6. Preview final workflow

In [ ]:
visualize_workflow(workflow)
workflow.to_dict()

### 4.7. Save workflow to collection

In [ ]:
from utils.api import get_or_create_workflow

workflow_id_or_dict = None

if SAVE_WF_TO_COLLECTION:
    saved_workflow_response = get_or_create_workflow(client.workflows, workflow, ACCOUNT_ID)
    saved_workflow = dict_to_namespace(saved_workflow_response)
    workflow_id_or_dict = saved_workflow._id
    print(f"Workflow ID: {saved_workflow._id}")
else:
    workflow_id_or_dict = workflow.to_dict()
    print("Workflow will be embedded into job")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create compute configuration

In [ ]:
from mat3ra.ide.compute import Compute

cluster = next((c for c in clusters if CLUSTER_NAME and CLUSTER_NAME in c["hostname"]), clusters[0] if clusters else None)

compute = Compute(
    cluster=cluster,
    queue=QUEUE_NAME,
    ppn=PPN
)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

## 6. Create the job
### 6.1. Create job

In [ ]:
from utils.api import create_job
from utils.visualize import display_JSON

print(f"Material: {saved_material._id}")
print(f"Workflow: {workflow_id_or_dict if SAVE_WF_TO_COLLECTION else '(embedded)'}")
print(f"Project: {project_id}")

job_response = create_job(
    jobs_endpoint=client.jobs,
    materials=[vars(saved_material)],
    workflow_id_or_dict=workflow_id_or_dict,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=JOB_NAME,
    compute=compute.to_dict(),
    save_to_collection=SAVE_WF_TO_COLLECTION,
)

job_dict = job_response[0]
job = dict_to_namespace(job_dict)
job_id = job._id
print("✅ Job created successfully!")
print(f"Job ID: {job_id}")
display_JSON(job_response)

## 7. Submit the job and monitor the status

In [ ]:
client.jobs.submit(job_id)
print(f"✅ Job {job_id} submitted successfully!")

In [ ]:
from utils.api import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, [job_id])

## 8. Retrieve and visualize results
### 8.1. Total Energy

In [ ]:
from mat3ra.prode import PropertyName
from utils.visualize import visualize_properties

total_energy_data = client.properties.get_for_job(job_id, property_name=PropertyName.scalar.total_energy.value)
visualize_properties(total_energy_data, title="Total Energy")

### 8.2. Wavefunction Amplitude

In [ ]:
if PP_PROPERTY == "wavefunction_amplitude":
    wfn_data = client.properties.get_for_job(
        job_id, property_name=PropertyName.non_scalar.wavefunction_amplitude.value
    )
    visualize_properties(wfn_data, title=f"Wavefunction Amplitude ({WFN_BAND}, plane: {WFN_PLANE})")

### 8.3. Electronic Density Mesh

In [ ]:
if PP_PROPERTY == "electronic_density_mesh":
    print("Electronic density mesh output (density.xsf) is available in the job files.")
    all_properties = client.properties.get_for_job(job_id)
    visualize_properties(all_properties, title="Electronic Density Mesh - Job Properties")

### 8.4. Average Electrostatic Potential

In [ ]:
if PP_PROPERTY == "average_electrostatic_potential":
    avg_esp_data = client.properties.get_for_job(
        job_id, property_name=PropertyName.non_scalar.average_potential_profile.value
    )
    visualize_properties(
        avg_esp_data,
        title=f"Average Electrostatic Potential (scale={AVG_SAMPLING_SCALE}, npts={AVG_NUM_POINTS}, window={AVG_SMOOTHING_WINDOW})"
    )

### 8.5. LDOS

In [ ]:
if PP_PROPERTY == "ldos":
    print(f"LDOS output (ldos.xsf) is available in the job files for energy range [{LDOS_EMIN}, {LDOS_EMAX}] eV.")
    all_properties = client.properties.get_for_job(job_id)
    visualize_properties(all_properties, title="LDOS - Job Properties")